<a href="https://colab.research.google.com/github/Andreas-Lukito/Stock_Sentiment_Analysis/blob/dev%2Fandreas/notebooks/07_BERT_categorical_new_data_oversampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT for Predicting News Sentiment

## Install Libraries

In [45]:
! pip install contractions emoji gensim optuna torch matplotlib
! pip install wandb

## Iport Libraries

In [46]:
# Common Python Libraries
import numpy as np
import pandas as pd
import os
import sys
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import random
from datasets import Dataset

# Deep Learning Libraries
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, EarlyStoppingCallback
from torch.optim import Adam, AdamW
from accelerate import Accelerator

# Data Preprocessing
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler

## Download nltk dependencies
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

# Model metrics
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, RocCurveDisplay, f1_score, precision_score, recall_score, accuracy_score
from itertools import cycle
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Model Tracking
import wandb

# Google Colab Setup
from google.colab import drive
drive.mount('/content/drive')
project_path = "/content/drive/MyDrive/stock_news_sentiment_analysis"

# project_path = "../"

# Project Seed for Reproducability
SEED = random.randint(0, 2**32 - 1)  # Random integer between 0 and 2^32-1
print(f"seed: {SEED}")

model_name = "google-bert/bert-base-uncased"

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
seed: 965377227


## Choose Device

In [47]:
# Detect available device
if torch.cuda.is_available():
    # check if ROCm backend is active
    if torch.version.hip is not None:
        backend = "ROCm"
    else:
        backend = "CUDA"

    device = torch.device("cuda")
    print(f"PyTorch is using GPU: {torch.cuda.get_device_name(0)}")
    print(f"Backend: {backend}")
else:
    device = torch.device("cpu")
    print("PyTorch is not using GPU — running on CPU")

PyTorch is using GPU: NVIDIA A100-SXM4-40GB
Backend: CUDA


## Import Data

In [48]:
before_date = "2025-11"

# Data path
categorized_data_path = os.path.join(project_path,f"news_cache/catgorized_data/categorized_news_data_rescraped_cleaned.csv")

# Import Data
news_data = pd.read_csv(filepath_or_buffer=categorized_data_path, sep=',')

In [49]:
news_data.head()

,index,uuid,title,description,keywords,snippet,url,image_url,language,published_at,source,relevance_score,entities,similar,sentiment,text,clean_text,categorical_sentiment_3_class,length
0,0,487e6a88-d3c2-4ae1-8dc2-26af6b31d688,2025: The Year Of Alphabet (GOOG),No stock has seen a bigger jump recently than ...,NaN,vzphotos/iStock Editorial via Getty Images\n\n...,https://seekingalpha.com/article/4848680-2025-...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:30:00.000000Z,seekingalpha.com,NaN,"[{'symbol': 'GOOGL', 'name': 'Alphabet Inc.', ...",[],0.0000,2025: The Year Of Alphabet (GOOG) No stock has...,2025 the year of alphabet goog no stock has se...,neutral,42
1,1,92b5c2bd-d324-4ae8-b115-2cfd95a8fa98,Why I'm Doubling Down On My Adobe Position (NA...,"Adobe's revenue is highly predictable, driven ...",NaN,To say that Adobe ( ADBE ) stock has not had a...,https://seekingalpha.com/article/4848762-why-i...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:25:01.000000Z,seekingalpha.com,NaN,"[{'symbol': 'ADBE', 'name': 'Adobe Inc.', 'exc...",[],0.0000,Why I'm Doubling Down On My Adobe Position (NA...,why i am doubling down on my adobe position na...,neutral,259
2,2,9084e5f1-75f5-4f15-aa3d-0676073b4aaf,Global week ahead: The start of a Santa Rally ...,NaN,"STOXX 600, business news",And just like that... December is upon us. It'...,https://www.cnbc.com/2025/11/30/global-week-ah...,https://image.cnbcfm.com/api/v1/image/10823257...,en,2025-11-30T05:10:58.000000Z,cnbc.com,NaN,"[{'symbol': 'M', 'name': ""Macy's, Inc."", 'exch...",[],0.6908,Global week ahead: The start of a Santa Rally ...,global week ahead the start of a santa rally o...,positive,493
3,3,487e6a88-d3c2-4ae1-8dc2-26af6b31d688,2025: The Year Of Alphabet (GOOG),No stock has seen a bigger jump recently than ...,NaN,vzphotos/iStock Editorial via Getty Images\n\n...,https://seekingalpha.com/article/4848680-2025-...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:30:00.000000Z,seekingalpha.com,NaN,"[{'symbol': 'GOOGL', 'name': 'Alphabet Inc.', ...",[],0.0000,2025: The Year Of Alphabet (GOOG) No stock has...,2025 the year of alphabet goog no stock has se...,neutral,42
4,4,92b5c2bd-d324-4ae8-b115-2cfd95a8fa98,Why I'm Doubling Down On My Adobe Position (NA...,"Adobe's revenue is highly predictable, driven ...",NaN,To say that Adobe ( ADBE ) stock has not had a...,https://seekingalpha.com/article/4848762-why-i...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:25:01.000000Z,seekingalpha.com,NaN,"[{'symbol': 'ADBE', 'name': 'Adobe Inc.', 'exc...",[],0.0000,Why I'm Doubling Down On My Adobe Position (NA...,why i am doubling down on my adobe position na...,neutral,259


In [50]:
news_data.isna().sum()

,0
index,0
uuid,0
title,0
description,3148
keywords,35210
snippet,216
url,0
image_url,299
language,0
published_at,0


In [51]:
news_data = news_data.dropna(subset=["sentiment"])

In [52]:
news_data.isna().sum()

,0
index,0
uuid,0
title,0
description,3148
keywords,35207
snippet,216
url,0
image_url,299
language,0
published_at,0


In [53]:
value_maps = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

news_data["categorical_sentiment_3_class_value_maps"] = news_data["categorical_sentiment_3_class"].map(value_maps)

## Apply Random Oversampling

In [54]:
df = pd.DataFrame({
    "text": news_data["text"],
    "label": news_data["categorical_sentiment_3_class_value_maps"]
})

rus = RandomOverSampler(random_state=SEED)
X_resampled, y_resampled = rus.fit_resample(df[["text"]], df["label"])

ros_dataframe = pd.DataFrame({
    "text": X_resampled["text"],
    "label": y_resampled
})

In [55]:
ros_dataframe.head()

,text,label
0,2025: The Year Of Alphabet (GOOG) No stock has...,1
1,Why I'm Doubling Down On My Adobe Position (NA...,1
2,Global week ahead: The start of a Santa Rally ...,2
3,2025: The Year Of Alphabet (GOOG) No stock has...,1
4,Why I'm Doubling Down On My Adobe Position (NA...,1


## Split the data to Train, Test, and Validation

In [56]:
test_size = 0.10
val_size = 0.50

# Splitting the data into train and temp (which will be further split into validation and test)
train_df, test_df = train_test_split(ros_dataframe, test_size=test_size, random_state=SEED)

# Splitting train into validation and test sets
val_df, test_df = train_test_split(test_df, test_size=val_size, random_state=SEED)

In [57]:
train_df.shape, test_df.shape, val_df.shape

((85004, 2), (4723, 2), (4722, 2))

In [58]:
train_df = train_df[["text", "label"]]
train_df = train_df.rename(columns = {"text": "text", "label":"labels"})

val_df = val_df[["text", "label"]]
val_df = val_df.rename(columns = {"text": "text", "label":"labels"})

test_df = test_df[["text","label"]]
test_df = test_df.rename(columns = {"text": "text", "label":"labels"})

## Data Preprocessing

### Convert Dataframe to Dataset for the Model Trainer

In [59]:
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

### Tokenizer for the text

In [60]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples, tokenizer=tokenizer):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=128   # keep this
    )

In [61]:
train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

Map:   0%|          | 0/85004 [00:00<?, ? examples/s]

Map:   0%|          | 0/4722 [00:00<?, ? examples/s]

Map:   0%|          | 0/4723 [00:00<?, ? examples/s]

In [62]:
# remove unwanted columns
train_ds = train_ds.remove_columns(["text", "__index_level_0__"])
val_ds = val_ds.remove_columns(["text", "__index_level_0__"])
test_ds = test_ds.remove_columns(["text", "__index_level_0__"])

## Train Model

### Model Arguments

In [63]:
learning_rate=2e-5
run_name = f"bert_categorical_new_data_{learning_rate}_oversampling"
model_checkpoint_directory = os.path.join(project_path, f"model_checkpoints/bert_categorical/{run_name}")
os.makedirs(model_checkpoint_directory, exist_ok=True)

training_args = TrainingArguments(
    output_dir=model_checkpoint_directory,
    report_to="wandb",
    run_name=run_name,

    eval_strategy="epoch",
    save_strategy="epoch",

    num_train_epochs=1000,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    learning_rate=learning_rate,
    lr_scheduler_type="linear",

    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,

    save_total_limit=2,

    seed=SEED
)

### Model Metrics for Training

In [64]:
def model_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()

    f1_weighted = f1_score(labels, predictions, average='weighted', zero_division=0)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)

    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    accuracy = accuracy_score(labels, predictions)

    try:
        roc_auc = roc_auc_score(labels, probs, multi_class='ovr')
    except:
        roc_auc = None

    return {
        "f1_score": f1_weighted,
        "macro_f1": f1_macro,
        "precision": precision,
        "recall": recall,
        "accuracy": accuracy,
        "roc_auc": roc_auc
    }

### wandb setup

In [65]:
wandb.login()

# set the wandb project where this run will be logged
os.environ["WANDB_PROJECT"]="Thesis-Stock-Sentiment-Analysis"

# save your trained model checkpoint to wandb
os.environ["WANDB_LOG_MODEL"]="true"

# turn off watch to log faster
os.environ["WANDB_WATCH"]="false"

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


### Model Trainer

In [66]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3, # Since there are three classes ["Negative", "Neutral", "Positive"]
    ignore_mismatched_sizes=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [67]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    compute_metrics = model_metrics
    ,callbacks=[EarlyStoppingCallback(early_stopping_patience=10)]
)

[transformers] Received unrecognized `WANDB_LOG_MODEL` setting value=true; so disabling `WANDB_LOG_MODEL`


In [68]:
train_ds.column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [ ]:
wandb.login()

# set the wandb project where this run will be logged
os.environ["WANDB_PROJECT"]="Thesis-Stock-Sentiment-Analysis"

# save your trained model checkpoints to wandb
os.environ["WANDB_LOG_MODEL"]="checkpoint"

# turn off watch to log faster
os.environ["WANDB_WATCH"]="false"

trainer.train()

model_save_path = os.path.join(model_checkpoint_directory, "final_model")
os.makedirs(model_save_path, exist_ok=True)
trainer.save_model(model_save_path)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Epoch,Training Loss,Validation Loss,F1 Score,Macro F1,Precision,Recall,Accuracy,Roc Auc
1,0.656336,0.622400,0.736387,0.736287,0.742801,0.734858,0.734858,0.891966
2,0.517907,0.534635,0.789080,0.788941,0.793521,0.790131,0.790131,0.923201
3,0.362794,0.520142,0.803675,0.803559,0.805469,0.805379,0.805379,0.933643
4,0.268342,0.538931,0.819523,0.819406,0.820126,0.820627,0.820627,0.940685
5,0.189184,0.618956,0.828587,0.828504,0.830578,0.828251,0.828251,0.940869
6,0.150333,0.706841,0.824293,0.824211,0.824447,0.825074,0.825074,0.939804
7,0.129126,0.859117,0.834636,0.834548,0.834852,0.835028,0.835028,0.940198
8,0.117585,0.790220,0.832529,0.832399,0.832581,0.832486,0.832486,0.941432
9,0.103721,0.979672,0.830479,0.830384,0.830702,0.830580,0.830580,0.938624
10,0.097883,0.897637,0.835920,0.835807,0.838688,0.835239,0.835239,0.942081


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Model Evaluation

### Model Evaluation Function

In [ ]:
def evaluate_model_with_trainer(trainer, dataset):
    predictions_output = trainer.predict(dataset)
    logits = predictions_output.predictions
    labels = predictions_output.label_ids
    metrics = predictions_output.metrics

    preds = np.argmax(logits, axis=-1)

    # Generate classification report
    report = classification_report(labels, preds, zero_division=0)

    # Generate confusion matrix
    cm = confusion_matrix(labels, preds)

    # For ROC AUC, we need probabilities. The model outputs logits, so convert to probabilities.
    # Make sure to handle multi_class correctly for 'ovr'
    try:
        probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
        roc_auc = roc_auc_score(labels, probs, multi_class='ovr')
    except ValueError: # Handle cases where roc_auc_score might fail (e.g., single class in evaluation)
        roc_auc = None

    return report, cm, roc_auc, labels, preds

### Get the Model

In [ ]:
trained_model = trainer.model

# Re-instantiate Trainer for evaluation to ensure Accelerator is in a valid state
evaluation_trainer = Trainer(
    model=trained_model, # Use the best model obtained after training
    args=training_args,  # Use the same training arguments
    eval_dataset=test_ds, # Only need the test dataset for testing
    compute_metrics=model_metrics # Keep the same metrics computation
)

In [ ]:
classification_report_str, confusion_matrix_val, roc_auc_val, labels, preds = evaluate_model_with_trainer(evaluation_trainer, test_ds)

In [ ]:
print("========== Classification Report ==========")
print(classification_report_str)

print("========== Confusion Matrix ==========")
print(confusion_matrix_val)

print()
print("========== ROC_AUC ==========")
print(roc_auc_val)

In [ ]:
def plot_roc_curve(y_test, y_pred):

  n_classes = len(np.unique(y_test))
  y_test = label_binarize(y_test, classes=np.arange(n_classes))
  y_pred = label_binarize(y_pred, classes=np.arange(n_classes))

  # Compute ROC curve and ROC area for each class
  fpr = dict()
  tpr = dict()
  roc_auc = dict()
  for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test[:, i], y_pred[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

  # Compute micro-average ROC curve and ROC area
  fpr["micro"], tpr["micro"], _ = roc_curve(y_test.ravel(), y_pred.ravel())
  roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

  # First aggregate all false positive rates
  all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

  # Then interpolate all ROC curves at this points
  mean_tpr = np.zeros_like(all_fpr)
  for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

  # Finally average it and compute AUC
  mean_tpr /= n_classes

  fpr["macro"] = all_fpr
  tpr["macro"] = mean_tpr
  roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

  # Plot all ROC curves
  #plt.figure(figsize=(10,5))
  plt.figure(dpi=600)
  lw = 2
  plt.plot(fpr["micro"], tpr["micro"],
    label="micro-average ROC curve (area = {0:0.2f})".format(roc_auc["micro"]),
    color="deeppink", linestyle=":", linewidth=4,)

  plt.plot(fpr["macro"], tpr["macro"],
    label="macro-average ROC curve (area = {0:0.2f})".format(roc_auc["macro"]),
    color="navy", linestyle=":", linewidth=4,)

  colors = cycle(["aqua", "darkorange", "darkgreen", "yellow", "blue"])
  for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=lw,
        label="ROC curve of class {0} (area = {1:0.2f})".format(i, roc_auc[i]),)

  plt.plot([0, 1], [0, 1], "k--", lw=lw)
  plt.xlim([0.0, 1.0])
  plt.ylim([0.0, 1.05])
  plt.xlabel("False Positive Rate")
  plt.ylabel("True Positive Rate")
  plt.title("Receiver Operating Characteristic (ROC) curve")
  plt.legend()

In [ ]:
plot_roc_curve(labels, preds)

In [ ]:
wandb.finish()